In [ ]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing
from typing import Callable


In [ ]:
import prism

In [ ]:
base_dir = "../IMAGE-Mat/scripts/vema"
preprocessing_results, preprocessing_xarray = preprocessing(base_dir)

In [ ]:
vehicle_nr = preprocessing_results['total_nr_vehicles_simple']
life_time_vehicles = preprocessing_results["lifetimes_vehicles"].to_xarray()

In [ ]:
from survival import ScipySurvival, convert_life_time_vehicles, SurvivalMatrix
from util import convert_vehicles
from stock import compute_historic, compute_dynamic_stock_driven
import numpy as np
       
lifetime_params = convert_life_time_vehicles(life_time_vehicles)
survival = ScipySurvival(lifetime_params)
survival_matrix = SurvivalMatrix(survival)
xr_vehicle_nr = convert_vehicles(vehicle_nr)

In [ ]:
start_simulation = 1970
end_simulation = xr_vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 27)])
Mode = prism.NewDim("mode", coords=list(vehicle_nr.columns.levels[0].unique()))
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr.index.to_numpy()])

In [ ]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: Callable    # defines the stock function to use e.g. stock or inflow driven

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                         self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.stock_by_cohort,  self.inflow, self.outflow_by_cohort, self.survival_matrix, t)

In [ ]:
timeline = prism.Timeline(start=xr_vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [ ]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix, stock=xr_vehicle_nr, stock_function = compute_dynamic_stock_driven)

In [ ]:
model.simulate(timeline_simulate)